In [37]:
setwd("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb")
library(argparse)
library(data.table)

# simple method to shuffle knockouts
shuffle_knockouts <- function(d){
    d$KO <- rbinom(n=nrow(d), size=1, prob = d$pTKO)
    d$pKO <- ifelse((d$KO == 1 & d$unphased_het == 0), 1,
         ifelse((d$KO == 1 & d$hom_alt_n > 0), 1,
         ifelse((d$phased_het == 1 & d$unphased_het > 0), 1 - 1*(1/2)^d$unphased_het,
         ifelse((d$phased_het == 0 & d$unphased_het > 1), 1 - 2*(1/2)^d$unphased_het, 0))))
    return(d$pKO)
}

# make header of VCF file
make_vcf_dosage_header <- function(chrom){
    vcf_format <- '##fileformat=VCFv4.2'
    vcf_entry <-  '##FORMAT=<ID=DS,Number=1,Type=Float,Description="">'
    vcf_filter <- '##FILTER=<ID=PASS,Description="All filters passed">"'
    vcf_contig <- paste0('##contig=<ID=',chrom,',length=81195210>')
    vcf_out <- paste(vcf_format, vcf_entry, vcf_filter, vcf_contig, sep = '\n')
    return(vcf_out)
}


# make a random string of n characters and numbers
random_string <- function(n){
    return(paste0(sample(c(letters, LETTERS), n, replace = TRUE), collapse = ""))
}

# make vcf-like rows for dosage entries
make_vcf_dosage_rows <- function(chrom, positions, marker, use_random_alleles = TRUE){
    rows <- length(positions)
    refs <- unlist(ifelse(use_random_alleles, list(replicate(rows, random_string(4))), "A0"))
    alts <- unlist(ifelse(use_random_alleles, list(replicate(rows, random_string(4))), "A1"))
    return(data.table(
      "#CHROM" = chrom,
      POS = positions,
      ID = marker,
      REF = refs,
      ALT = alts,
      QUAL = '.',
      FILTER = '.',
      INFO = '.',
      FORMAT = 'DS'
    ))
}

# read real variant data in long format (exported from hail)
format_real_variant_long_to_wide <- function(dt){
    
    stopifnot("s" %in% colnames(dt))
    stopifnot("locus" %in% colnames(dt))
    stopifnot("alleles" %in% colnames(dt))
    
    mapping <- dt[,c("locus","alleles")]
    mapping <- mapping[!duplicated(mapping),]
    
    # create mapping rows that are to be combined with actual dosages
    alleles <- as.data.frame(do.call(rbind, strsplit(gsub('(")|(\\])|(\\[)','',mapping$alleles), split = ',')))
    colnames(alleles) <- c("REF","ALT")
    mapping <- cbind(mapping, alleles)
    mapping$chroms <- stringr::str_extract(mapping$locus, "chr[0-9]+")
    mapping$positions <- gsub("chr[0-9]+\\:", "",mapping$locus)
    mapping$marker <- paste0(mapping$locus, ":", mapping$REF, ":", mapping$ALT)
    stopifnot(length(unique(mapping$positions)) == length(mapping$positions)) # can't handle SNPs at same pos
    mapping_rows <- data.table(
        "#CHROM" = mapping$chroms,
          POS = mapping$positions,
          ID = mapping$marker,
          REF = mapping$REF,
          ALT = mapping$ALT,
          QUAL = '.',
          FILTER = '.',
          INFO = '.',
          FORMAT = 'DS',
          locus = mapping$locus
    )
    
    # go from long to wide format
    dt <- dt[,c('locus','s','DS')]
    dt <- data.table::dcast(locus~s, data = dt, value.var = "DS")
    
    # match mapping rows with dt rows
    new_index <- match(dt$locus, mapping$locus)
    mapping_rows <- mapping_rows[new_index,]
    mapping_rows$locus <- NULL
    
    #return(cbind(mapping_rows, dt))
    return(list(rows = mapping_rows, dosage = dt))
    
}


main <- function(args){

    #print(args)
    stopifnot(file.exists(args$input_path))
    stopifnot(!is.na(as.numeric(args$permutations)))
    stopifnot(!is.null(args$permutations))

    # seed for reproducibility
    seed <- as.numeric(args$seed)
    set.seed(seed)

    # replicate knockout
    n <- as.numeric(args$permutations)
    d <- fread(args$input_path)
    stopifnot(nrow(d) > 0)
    reps <- replicate(n, shuffle_knockouts(d))
    rownames(reps) <- d$s
    reps <- data.table(t(reps))

    # convert to dosage
    dosage <- reps * 2

    # only keep non-probabilistic knockouts
    if (args$only_non_prob_ko) dosage[dosage < 2] <- 0

    # combine synthethic rows with knockout matrix
    rows <- make_vcf_dosage_rows(args$chrom, 1:n, args$vcf_id)
    final <- cbind(rows, dosage)

    # (1) write header of VCF
    vcf_out = make_vcf_dosage_header(args$chrom)
    outfile = paste0(args$out_prefix, ".vcf")
    writeLines(text = vcf_out, outfile)

    # (2) append with permuted data
    fwrite(final, outfile, append = TRUE, quote = FALSE, sep = '\t', col.names = TRUE)

}

# add arguments
#parser <- ArgumentParser()
#parser$add_argument("--chrom", default=NULL, help = "chromosome")
#parser$add_argument("--input_path", default=NULL, help = "path to the input")
#parser$add_argument("--permutations", default=NULL, help = "number of times the gene should be permuted")
#parser$add_argument("--only_non_prob_ko", action="store_true", default=FALSE, help = "Only keep knockouts of phased heterozygotes.")
#parser$add_argument("--seed", default=NULL, help = "seed for randomizer")
#parser$add_argument("--vcf_id", default="GENE", help = "Substitute for rsid")
#parser$add_argument("--out_prefix", default=NULL, help = "prefix for out file")
#args <- parser$parse_args()

In [29]:
args <- list(
    input_path = "data/permute/genes/chr12/ukb_eur_wes_200k_pLoF_damaging_missense_chr12_ENSG00000010295.tsv.gz",
    permutations = 100,
    seed = 42,
    only_non_prob_ko = FALSE,
    vcf_id = "GENE",
    chrom = "chr2"
    
)

In [81]:
#print(args)
stopifnot(file.exists(args$input_path))
stopifnot(!is.na(as.numeric(args$permutations)))
stopifnot(!is.null(args$permutations))

# seed for reproducibility
seed <- as.numeric(args$seed)
set.seed(seed)

# replicate knockout
n <- as.numeric(args$permutations)
d <- fread(args$input_path)
stopifnot(nrow(d) > 0)
reps <- replicate(n, shuffle_knockouts(d))
rownames(reps) <- d$s
reps <- data.table(t(reps))

# convert to dosage
dosage <- reps * 2

# only keep non-probabilistic knockouts
if (args$only_non_prob_ko) dosage[dosage < 2] <- 0


In [82]:
# load real conditioning variants
cond_dt <- fread(path)
cond_dt$chr <- stringr::str_extract(cond_dt$locus, "chr[0-9]+")
cond_dt <- cond_dt[cond_dt$chr %in% args$chrom]
n_real_markers <- length(unique(cond_dt$locus))


In [83]:
if (n_real_markers > 0 & ){

    # ensure that samples are overlapping
    sample_overlap <- unique(intersect(cond_dt$s, d$s))

    # subset dosage matrix (with permuted phased)
    rows <- make_vcf_dosage_rows(args$chrom, 1:n, args$vcf_id)
    dosage <- dosage[,colnames(dosage) %in% sample_overlap, with = FALSE]
    rows_dosage <- cbind(rows, dosage)
    
    # subset real dosage matrix (with actual calls/DS)
    cond_dt <- cond_dt[cond_dt$s %in% sample_overlap,]
    cond_lst <- format_real_variant_long_to_wide(cond_dt)
    
    # get long format
    cond_rows <- cond_lst$rows
    cond_dosage <- cond_lst$dosage
    
    # match columns
    cond_dosage$locus <- NULL
    cond_dosage <- cond_dosage[,colnames(dosage), with = FALSE]
    
    # combine columns and rows
    cond_rows_dosage <- cbind(cond_rows, cond_dosage)
    final <- rbind(cond_rows_dosage, rows_dosage)
    
}  else {
    
    rows <- make_vcf_dosage_rows(args$chrom, 1:n, args$vcf_id)
    rows_dosage <- cbind(rows, dosage)
    final <- rows_dosage
    
}

In [85]:
head(final, n = 5)

#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,1000028,⋯,6026237,6026268,6026270,6026295,6026311,6026322,6026335,6026383,6026397,6026401
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
chr2,20126916,chr2:20126916:T:C,T,C,.,.,.,DS,1,⋯,1,2,2,2,0,0,0,1,1,0
chr2,26786610,chr2:26786610:C:T,C,T,.,.,.,DS,0,⋯,1,1,1,1,1,0,0,0,0,0
chr2,1,GENE,WKay,YRsf,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0
chr2,2,GENE,jJrW,RyDP,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0
chr2,3,GENE,UxgJ,MYyb,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0


In [74]:
head(rows_dosage, n = 2)

#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,1000028,⋯,6026237,6026268,6026270,6026295,6026311,6026322,6026335,6026383,6026397,6026401
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
chr2,1,GENE,WKay,YRsf,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0
chr2,2,GENE,jJrW,RyDP,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0


In [75]:
tes <- rbind(cond_rows_dosage, cond_rows)

ERROR: Error in rbindlist(l, use.names, fill, idcol): Item 2 has 9 columns, inconsistent with item 1 which has 176603 columns. To fill missing columns use fill=TRUE.


In [8]:
path = "data/conditional/common/marker_mt/conditional_markers_chrundefined.tsv.gz"
cond_dt <- fread(path)
cond_dt <- cond_dt[cond_dt$s %in% d$s,]



cond_dosage <- format_real_variant_long_to_wide(cond_dt)

In [9]:
length(unique(cond_dt$s)) 

[1] 176594

In [6]:
sum(cond_dt$s %in% d$s) 
sum(d$s %in% cond_dt$s)

[1] 0

[1] 0

In [7]:
length(unique(cond_dt$s)) #  176915
length(unique(d$s))
176924 - 9

[1] 0

[1] 176915

[1] 176915